## qt Workspace

In [1]:
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from data.dataset import PretrainDataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace, Digits, Sequence, Punctuation

### Make Tokenizer

In [ ]:
# make full train data into .txt files in temp dir for tokenization, preprocess etc.
for file_name in os.listdir('data/train'):
    df = pd.read_parquet(f'data/train/{file_name}')
    df = df[df['language'] == 'en']
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].str.replace(r"[^a-z0-9 [:space:][:punct:]]", '', regex=True)
    df['text'].to_csv(f'data/temp/{file_name[:-8]}.txt', index=False, header=False)

In [10]:
# train tokenizer on all training text

VOCAB_SIZE = 13_000

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[EOS]", "[USER]", "[AI]"], vocab_size=VOCAB_SIZE)

tokenizer.pre_tokenizer = Sequence([Whitespace(), Digits(individual_digits=True), Punctuation()])

files = [f"data/temp/{i}" for i in os.listdir('data/temp')]
# files = [f'data/013_00000.txt']
tokenizer.train(files, trainer)

tokenizer.save("data/tokenizer-wiki.json")


In [4]:
tokenizer = Tokenizer.from_file("data/tokenizer-wiki.json")

output = tokenizer.encode("Hello, y'all! How are you 😁 ? I like the number 8675309 and 911 and 200 and 100 and 40 and 8".lower())
print(output.tokens)
print(output.ids)

['hel', 'lo', ',', 'y', "'", 'all', '!', 'how', 'are', 'you', '😁', '?', 'i', 'like', 'the', 'number', '8', '6', '7', '5', '3', '0', '9', 'and', '9', '1', '1', 'and', '2', '0', '0', 'and', '1', '0', '0', 'and', '4', '0', 'and', '8']
[7718, 7439, 14, 65, 9, 7498, 3, 7619, 7447, 7461, 7296, 33, 49, 7686, 7403, 8139, 26, 24, 25, 23, 21, 18, 27, 7418, 27, 19, 19, 7418, 20, 18, 18, 7418, 19, 18, 18, 7418, 22, 18, 7418, 26]
